# S1 · 02 · Herramientas y acciones
Tools, `ModelRetry` y aprobación. Las secciones 1 a 5 son ejemplos mínimos de las **Demos 4 y 5**; tu trabajo son los **Ejercicios 3 a 7** (≈ 40 min; el 7 completa tu `mi_copiloto/`).

Corre sin red ni LLM (`COURSE_OFFLINE=1`, por defecto). Datos ficticios: «hoy» es el martes 14 de mayo de 2030.


## Preparación
Igual que en el notebook 01. Las piezas ya escritas del proyecto (`Deps`, `Respuesta`, `consultar_agenda`, `buscar_correos`, `clima`) viven en `mi_copiloto/agente.py`: ábrelo, es código de verdad. Al final del notebook completarás lo que falta en ese archivo.


In [ ]:
import json

from utilidades.config import crear_modelo, describir_ruta
from utilidades.practica import preparar, revisar, ver_mensajes

from mi_copiloto.agente import (
    INSTRUCCIONES,
    PLANTILLA_JSON,
    Deps,
    Respuesta,
    buscar_correos,
    clima,
    consultar_agenda,
    fecha_de_hoy,
)

settings, OFFLINE = preparar()
modelo = crear_modelo(settings, OFFLINE)
print("Modelo activo:", describir_ruta(settings, OFFLINE))


## 1 · Una tool es una función con docstring
El modelo **no ve tu código**: ve un nombre, una descripción y un **JSON Schema**. Pydantic AI los saca de la firma (tipos y valores por defecto) y del docstring (primera línea y sección `Args:`). Mira lo que ve el modelo de `buscar_correos`.


In [ ]:
from pydantic_ai import Tool

contrato = Tool(buscar_correos).tool_def
print(contrato.name, "·", contrato.description)
print(json.dumps(contrato.parameters_json_schema, ensure_ascii=False, indent=1))


Observa tres cosas: `texto` y `limite` **no** son obligatorios porque la firma trae valores por defecto; el tipo `int` impide que el modelo mande «tres»; y el docstring es el **prompt de la tool**. `ctx` no aparece: es del programa, no del modelo. Cambia el docstring en `mi_copiloto/agente.py`, reinicia el kernel y mira cómo cambia lo que ve el modelo.
## 2 · Un agente con tools de lectura
`Deps` es lo que el agente necesita del mundo (la carpeta de datos, qué día es hoy); lo pone el programa en cada `run`. Con Ollama y tools, el esquema de `Respuesta` va **dentro del prompt** (`PromptedOutput`). Y como una escritura puede quedar esperando permiso, `output_type` es una **lista**: la respuesta final o un `DeferredToolRequests`.


In [ ]:
from pydantic_ai import Agent, DeferredToolRequests, PromptedOutput

salida = (
    PromptedOutput(Respuesta, template=PLANTILLA_JSON)  # la plantilla pide consultar las tools primero
    if getattr(modelo, "system", "") == "ollama"
    else Respuesta
)

agente = Agent(
    modelo,
    deps_type=Deps,
    output_type=[salida, DeferredToolRequests],
    instructions=[INSTRUCCIONES, fecha_de_hoy],
    tools=[consultar_agenda, buscar_correos, clima],
)
r = await agente.run("¿Qué tengo en la agenda el 2030-05-14?", deps=Deps(sin_red=OFFLINE))
print(r.output.texto, "| fuentes:", r.output.fuentes)
ver_mensajes(r)


Para ver el contrato completo que recibe el modelo usamos un «espía»: un modelo falso que anota las tools y no ejecuta nada.


In [ ]:
from pydantic_ai.messages import ModelResponse, TextPart, ToolCallPart
from pydantic_ai.models.function import AgentInfo, FunctionModel

visto = {}


def espiar(mensajes, info: AgentInfo) -> ModelResponse:
    visto["tools"] = info.function_tools
    if info.output_tools:  # salida por tool; con Ollama (PromptedOutput) la respuesta es texto JSON
        return ModelResponse(parts=[ToolCallPart(info.output_tools[0].name, {"texto": "fin"})])
    return ModelResponse(parts=[TextPart('{"texto": "fin"}')])


with agente.override(model=FunctionModel(espiar)):
    await agente.run("hola", deps=Deps())
for t in visto["tools"]:
    print(t.name, "·", t.description.splitlines()[0], "·", list(t.parameters_json_schema["properties"]))


## 3 · Tools que fallan bien: `ModelRetry`
`consultar_agenda` valida la fecha y, si es imposible, lanza `ModelRetry("mensaje")`: el mensaje **vuelve al modelo** como pista y este reintenta. Probemos con una fecha imposible: el simulador no corrige (solo informa), pero con IA real el segundo intento suele traer la fecha bien escrita. Busca la línea «reintento». (Un modelo listo puede notar que el 30 de febrero no existe y responder sin llamar la tool: también es válido; con el simulador el reintento se ve siempre.)


In [ ]:
r = await agente.run("¿Qué tengo en la agenda el 2030-02-30?", deps=Deps(sin_red=OFFLINE))
ver_mensajes(r, 200)


## 4 · APIs públicas sin clave
`clima` llama a **Open-Meteo** con `httpx`: `timeout`, `raise_for_status` (un error 500 no es un dato) y una salida compacta con `url`, la **fuente que el agente puede citar**. Si no hay red, usa una respuesta guardada y lo avisa con `simulado: true`. Si una tool no devolviera `fuente` o `url`, el agente no podría citarla. La tool es `async`: se llama con `await` (aquí, a través del agente).


In [ ]:
for consulta in ["¿Llevo paraguas a Tarata?", "¿Qué correos tengo de Ana?"]:
    r = await agente.run(consulta, deps=Deps(sin_red=OFFLINE))
    print("→", r.output.texto[:200])
    print("   fuentes:", r.output.fuentes, "\n")


## 5 · Lectura frente a escritura: acciones con aprobación
Las tools de lectura se ejecutan solas. La de escritura lleva `requires_approval=True` (envolviendo la función en `Tool(...)`): el modelo puede **pedirla**, pero el run **termina** con un `DeferredToolRequests` y nada se guarda. La decisión la toma una persona **fuera del modelo**; con `DeferredToolResults` reanudamos el mismo run.


In [ ]:
from pydantic_ai import DeferredToolResults, RunContext, ToolDenied


def crear_evento(ctx: RunContext[Deps], titulo: str, fecha: str, inicio: str, fin: str) -> dict:
    """Crea un evento en la agenda de Alex. Requiere aprobación de la persona.

    Args:
        titulo: Nombre del evento.
        fecha: Día en formato AAAA-MM-DD.
        inicio: Hora de inicio HH:MM (24 horas).
        fin: Hora de fin HH:MM (24 horas).
    """
    evento = {"fecha": fecha, "inicio": inicio, "fin": fin, "titulo": titulo, "tipo": "nuevo"}
    ctx.deps.eventos_nuevos.append(evento)
    return {**evento, "fuente": f"agenda:{fecha}"}


agente = Agent(
    modelo,
    deps_type=Deps,
    output_type=[salida, DeferredToolRequests],
    instructions=[INSTRUCCIONES, fecha_de_hoy],
    tools=[
        consultar_agenda,
        buscar_correos,
        clima,
        Tool(crear_evento, requires_approval=True),  # escribe: el modelo la pide, una persona decide
    ],
)
deps = Deps(sin_red=OFFLINE)
r = await agente.run("Agenda «Llamada con Ana» el 2030-05-16 de 15:00 a 16:00", deps=deps)
print("Salida del run:", type(r.output).__name__)
if isinstance(r.output, DeferredToolRequests):
    for llamada in r.output.approvals:
        print("Pide:", llamada.tool_name, llamada.args_as_dict())
else:  # un modelo pequeño o el simulador a veces responde sin proponer la tool: es un dato
    print("El modelo no propuso el evento esta vez. Repite el pedido o prueba con otro modelo.")
print("Eventos guardados todavía:", deps.eventos_nuevos)


Ahora **tú** decides. Aprobar ejecuta la tool y el agente continúa con su resultado. Rechazar (`ToolDenied`) la deniega sin ejecutarla: el modelo se entera de que la persona dijo no.


In [ ]:
if isinstance(r.output, DeferredToolRequests):
    decisiones = {c.tool_call_id: True for c in r.output.approvals}  # o ToolDenied("No")
    r_final = await agente.run(
        None,
        message_history=r.all_messages(),
        deferred_tool_results=DeferredToolResults(approvals=decisiones),
        deps=deps,
    )
    print(r_final.output.texto)
print("Eventos guardados ahora:", deps.eventos_nuevos)


## Ejercicio 3 · Una función pura y una tool nueva
**Se apoya en:** Demo 4 · **Nivel:** medio · ≈ 12 min. **Qué cambia:** en la demo las tools leían datos tal cual; aquí calculas algo nuevo (con cruces de horarios) y lo registras tú.

Alex pregunta «¿cuánto tiempo libre tengo?». Escribe `minutos_libres(fecha)`: suma los minutos libres de un día entre 08:00 y 20:00 a partir de los eventos de `agenda.json`. Una función pura se prueba sin modelo ni red: así separas un fallo de tu lógica de uno de selección del modelo. Cuidado con los eventos que se cruzan (el 16 de mayo): no cuentes dos veces el mismo tiempo ocupado.
Después regístrala como tool con `@agente_tiempo.tool_plain`. Con un modelo guionizado (`FunctionModel`) comprobamos que el modelo **ve** la tool, que el runtime la **ejecuta** y que el resultado **vuelve** al contexto. Recuerda: el docstring es el prompt de la tool y los errores de datos se devuelven con `ModelRetry`.


In [ ]:
from datetime import date

from pydantic_ai import ModelRetry
from pydantic_ai.messages import ModelRequest, ToolReturnPart

EVENTOS = json.loads((Deps().datos / "agenda.json").read_text(encoding="utf-8"))


def a_minutos(hora: str) -> int:
    horas, minutos = hora.split(":")
    return int(horas) * 60 + int(minutos)


def minutos_libres(fecha: str) -> int:
    """Minutos libres de un día entre 08:00 y 20:00. ValueError si la fecha no es AAAA-MM-DD."""
    date.fromisoformat(fecha)
    # TODO Ejercicio 3: toma los eventos del día (EVENTOS) y suma los minutos ocupados sin contar dos veces los
    # cruces. Pista: a_minutos("09:30") convierte una hora en minutos desde la medianoche.
    return 0


def guion_tiempo(mensajes, info: AgentInfo) -> ModelResponse:
    visto_e3["tools"] = [t.name for t in info.function_tools]
    ultimo = mensajes[-1]
    if isinstance(ultimo, ModelRequest) and any(isinstance(p, ToolReturnPart) for p in ultimo.parts):
        return ModelResponse(parts=[TextPart("Te quedan minutos libres.")])
    return ModelResponse(parts=[ToolCallPart("minutos_libres_dia", {"fecha": "2030-05-14"})])


visto_e3 = {}
agente_tiempo = Agent(FunctionModel(guion_tiempo))
# TODO Ejercicio 3: registra con @agente_tiempo.tool_plain una tool `minutos_libres_dia(fecha)` con docstring que
# devuelva {"fecha", "minutos_libres", "fuente"}. Si la fecha es inválida, lanza ModelRetry con un
# mensaje que diga el formato esperado.


In [ ]:
revisar("Ejercicio 3 · día con tres eventos (14 may)", lambda: minutos_libres("2030-05-14") == 540, "Son 720 min menos 180 ocupados")
revisar("Ejercicio 3 · eventos que se cruzan (16 may)", lambda: minutos_libres("2030-05-16") == 600, "No cuentes dos veces el cruce de 10:30 a 11:00")
revisar("Ejercicio 3 · día sin eventos", lambda: minutos_libres("2030-05-18") == 720, "Un día vacío tiene 720 minutos libres")


def rechaza_fecha_invalida() -> bool:
    try:
        minutos_libres("mañana")
    except ValueError:
        return True
    return False


revisar("Ejercicio 3 · rechaza «mañana»", rechaza_fecha_invalida, "Lanza ValueError con una fecha que no es AAAA-MM-DD")


In [ ]:
try:
    resultado_e3 = await agente_tiempo.run("¿Cuánto tiempo libre tengo hoy?")
except Exception as exc:  # la tool aún no existe: el guion pide una tool desconocida
    resultado_e3 = None
    print("Todavía no se puede ejecutar:", type(exc).__name__)


def retornos_e3() -> list:
    return [p for m in resultado_e3.all_messages() for p in m.parts if isinstance(p, ToolReturnPart)]


revisar("Ejercicio 3 · el modelo ve la tool", lambda: "minutos_libres_dia" in visto_e3["tools"], "Decora la función con @agente_tiempo.tool_plain")
revisar("Ejercicio 3 · la tool se ejecutó", lambda: retornos_e3()[0].content["minutos_libres"] == 540, "Devuelve un dict con la clave minutos_libres")
revisar("Ejercicio 3 · incluye fuente", lambda: "fuente" in retornos_e3()[0].content, "Añade fuente al resultado para poder citarlo")


## Ejercicio 4 · Rediseña una tool mal hecha
**Se apoya en:** Demo 4 · **Nivel:** medio · ≈ 8 min. **Qué cambia:** en la demo la tool ya estaba bien diseñada; aquí partes de una mala y la mejoras.

`correos_mal` tiene tres defectos: un nombre y docstring que no dicen nada, parámetros sin tipo y una salida enorme sin `fuente` (cada correo completo, con su cuerpo, que además puede traer instrucciones ajenas). Escribe `buscar_correos_bien(consulta, limite)`: docstring claro con sección `Args:`, filtro por palabra, `limite` respetado y, por cada correo, solo `id`, `de`, `asunto`, `extracto` (90 caracteres) y `fuente` (`correo:<id>`).


In [ ]:
CORREOS = json.loads((Deps().datos / "correo.json").read_text(encoding="utf-8"))


def correos_mal(q=""):
    """Hace cosas con correo."""
    return CORREOS


def buscar_correos_bien(consulta: str = "", limite: int = 3) -> list[dict]:
    # TODO Ejercicio 4: escribe un docstring claro con sección Args:, filtra por `consulta`, respeta `limite`
    # y devuelve por correo solo id, de, asunto, extracto (90 caracteres) y fuente="correo:<id>".
    """Correos."""
    return CORREOS


In [ ]:
def tamano(datos) -> int:
    return len(json.dumps(datos, ensure_ascii=False))


print("Salida mala:", tamano(correos_mal()), "caracteres; salida tuya:", tamano(buscar_correos_bien()))
revisar("Ejercicio 4 · respeta limite", lambda: len(buscar_correos_bien("", limite=2)) == 2, "Devuelve como máximo `limite` correos")
revisar("Ejercicio 4 · cada correo trae fuente", lambda: all(c["fuente"].startswith("correo:") for c in buscar_correos_bien()), "Añade fuente=f'correo:{id}'")
revisar("Ejercicio 4 · sin cuerpo completo", lambda: all("cuerpo" not in c and len(c["extracto"]) <= 90 for c in buscar_correos_bien()), "Devuelve extracto de 90 caracteres, no el cuerpo")
revisar("Ejercicio 4 · salida compacta", lambda: tamano(buscar_correos_bien()) < 0.5 * tamano(correos_mal()), "Menos de la mitad del tamaño de la salida mala")
revisar("Ejercicio 4 · docstring útil", lambda: "Args:" in (buscar_correos_bien.__doc__ or "") and len(buscar_correos_bien.__doc__) > 60, "Explica qué hace y documenta cada parámetro en Args:")


## Ejercicio 5 · Un error que enseña: `ModelRetry`
**Se apoya en:** Demo 4 (paso 4.4) · **Nivel:** básico · ≈ 6 min. **Qué cambia:** otra tool (horas, no fechas) y tú escribes el mensaje que guía al modelo.

`normalizar_hora` devuelve la hora como `HH:MM`, pero hoy lanza un `ValueError` seco que el modelo no puede usar. Cámbialo por un `ModelRetry` **accionable**: di qué recibiste, qué formato esperas y da un ejemplo. La autocorrección ejecuta un modelo guionizado que primero se equivoca («7pm») y luego corrige: solo corrige si tu mensaje llega de vuelta como reintento.


In [ ]:
def normalizar_hora(hora: str) -> str:
    """Devuelve la hora en formato HH:MM de 24 horas. `hora` debe venir como HH:MM."""
    partes = hora.split(":")
    if len(partes) == 2 and all(p.isdigit() for p in partes):
        h_, m_ = int(partes[0]), int(partes[1])
        if 0 <= h_ < 24 and 0 <= m_ < 60:
            return f"{h_:02d}:{m_:02d}"
    # TODO Ejercicio 5: sustituye este error seco por un ModelRetry accionable (qué recibiste, qué formato
    # esperas y un ejemplo).
    raise ValueError("hora inválida")


In [ ]:
from pydantic_ai.messages import RetryPromptPart


def guion_hora(mensajes, info: AgentInfo) -> ModelResponse:
    ultimo = mensajes[-1]
    if isinstance(ultimo, ModelRequest) and any(isinstance(p, ToolReturnPart) for p in ultimo.parts):
        return ModelResponse(parts=[TextPart("Hora normalizada.")])
    if isinstance(ultimo, ModelRequest) and any(isinstance(p, RetryPromptPart) for p in ultimo.parts):
        return ModelResponse(parts=[ToolCallPart("normalizar_hora", {"hora": "19:00"})])
    return ModelResponse(parts=[ToolCallPart("normalizar_hora", {"hora": "7pm"})])


agente_hora = Agent(FunctionModel(guion_hora), retries=2)
agente_hora.tool_plain(normalizar_hora)
try:
    resultado_e5 = await agente_hora.run("Pon la reunión a las 7 de la noche")
    ver_mensajes(resultado_e5)
except Exception as exc:
    resultado_e5 = None
    print("Todavía no se puede ejecutar:", type(exc).__name__)


def error_de_normalizar() -> Exception:
    try:
        normalizar_hora("7pm")
    except Exception as exc:
        return exc
    raise AssertionError("normalizar_hora('7pm') debía fallar")


def reintentos_e5() -> list:
    return [p for m in resultado_e5.all_messages() for p in m.parts if isinstance(p, RetryPromptPart)]


revisar("Ejercicio 5 · lanza ModelRetry", lambda: isinstance(error_de_normalizar(), ModelRetry), "Cambia el ValueError por ModelRetry")
revisar("Ejercicio 5 · el mensaje dice el formato", lambda: "HH:MM" in str(error_de_normalizar()), "Menciona el formato esperado: HH:MM")
revisar("Ejercicio 5 · repite lo recibido", lambda: "7pm" in str(error_de_normalizar()), "Incluye el valor recibido para que el modelo vea su error")
revisar("Ejercicio 5 · el modelo corrige y termina", lambda: len(reintentos_e5()) == 1 and "19:00" in str(resultado_e5.all_messages()), "El reintento llega al modelo y la segunda llamada funciona")


## Ejercicio 6 · Sin permiso no hay acción
**Se apoya en:** Demo 5 · **Nivel:** medio · ≈ 8 min. **Qué cambia:** en la demo solo se aprobó; aquí rechazas **y** apruebas, y mides lo que se guardó en cada caso.

La aprobación es la frontera entre «el modelo propone» y «el programa dispone». Escribe `probar_aprobacion(agente, crear_deps)`:
1. pide «Agenda una llamada» y anota si el run terminó con un `DeferredToolRequests`;
2. **rechaza** esa solicitud (`DeferredToolResults(approvals={id: ToolDenied("No")})`, reanudando con `message_history=r.all_messages()` y `deferred_tool_results=`) y cuenta los eventos guardados (`len(deps.eventos_nuevos)`);
3. pide lo mismo otra vez, **aprueba** la nueva solicitud (`True`) y vuelve a contar.

Devuelve `{"pendiente": bool, "tras_rechazo": int, "tras_aprobacion": int}`. Lo esperado: `True`, `0` y `1`. Es lo mismo que haces en la terminal al responder `N` y luego `s`. El agente de la prueba usa un modelo guionizado que siempre pide `crear_evento`, para que el ejercicio no dependa del modelo.


In [ ]:
def guion_evento(mensajes, info: AgentInfo) -> ModelResponse:
    ultimo = mensajes[-1]
    if isinstance(ultimo, ModelRequest) and any(isinstance(p, ToolReturnPart) for p in ultimo.parts):
        return ModelResponse(parts=[ToolCallPart(info.output_tools[0].name, {"texto": "Evento creado."})])
    args = {"titulo": "Llamada", "fecha": "2030-05-16", "inicio": "15:00", "fin": "16:00"}
    return ModelResponse(parts=[ToolCallPart("crear_evento", args)])


agente_prueba = Agent(
    FunctionModel(guion_evento),
    deps_type=Deps,
    output_type=[Respuesta, DeferredToolRequests],
    tools=[Tool(crear_evento, requires_approval=True)],
)


async def probar_aprobacion(agente, crear_deps) -> dict:
    deps = crear_deps()
    r = await agente.run("Agenda una llamada", deps=deps)
    pendiente = isinstance(r.output, DeferredToolRequests)
    if not pendiente:  # con IA real, un modelo pequeño a veces no propone la tool
        return {"pendiente": False, "tras_rechazo": -1, "tras_aprobacion": -1}
    # TODO Ejercicio 6: rechaza r.output.approvals[0].tool_call_id con ToolDenied, reanuda y cuenta los eventos.
    tras_rechazo = -1
    # TODO Ejercicio 6: pide de nuevo, aprueba la nueva solicitud con True, reanuda y vuelve a contar.
    tras_aprobacion = -1
    return {"pendiente": pendiente, "tras_rechazo": tras_rechazo, "tras_aprobacion": tras_aprobacion}


In [ ]:
resultado_e6 = await probar_aprobacion(agente_prueba, Deps)
print(resultado_e6)
revisar("Ejercicio 6 · la escritura queda pendiente", lambda: resultado_e6["pendiente"] is True, "El run debe terminar con DeferredToolRequests")
revisar("Ejercicio 6 · rechazar no guarda nada", lambda: resultado_e6["tras_rechazo"] == 0, "Reanuda con ToolDenied: la tool no se ejecuta")
revisar("Ejercicio 6 · aprobar guarda un evento", lambda: resultado_e6["tras_aprobacion"] == 1, "Pide otra vez y aprueba la nueva solicitud")


## Ejercicio 7 · Lleva tu agente a `mi_copiloto/`
**Se apoya en:** Demo 5 (paso 5.3) · **Nivel:** básico · ≈ 6 min. **Qué cambia:** pasas del notebook a un archivo del proyecto que luego usas desde la terminal.

Abre `mi_copiloto/agente.py` y completa los dos `# TODO Ejercicio 7` con lo que escribiste arriba:
1. el cuerpo de `crear_evento` (sección 5);
2. `crear_agente(modelo)`: el `Agent` con las cuatro tools, la de escritura con `requires_approval=True` (sección 5).

Luego conversa con tu agente en una terminal (desde la raíz del repo; añade `--simulado` si no tienes modelo):
```bash
uv run python -m mi_copiloto.chat
tú › ¿Qué tengo el 2030-05-14?
tú › Agenda «Llamada con Ana» el 2030-05-16 de 15:00 a 16:00     → ¿Apruebas crear_evento(...)? [s/N]
```
La celda siguiente comprueba tu archivo sin romper nada.


In [ ]:
import importlib

import mi_copiloto.agente as archivo

archivo = importlib.reload(archivo)  # si editaste el archivo, recoge los cambios
deps_chequeo = archivo.Deps()
agente_archivo = archivo.crear_agente(FunctionModel(guion_evento))


def tools_del_archivo() -> dict:
    return agente_archivo._function_toolset.tools


def crear_evento_directo() -> int:
    ctx = type("Ctx", (), {"deps": deps_chequeo})()
    archivo.crear_evento(ctx, "Prueba", "2030-05-16", "15:00", "16:00")
    return len(deps_chequeo.eventos_nuevos)


revisar("Ejercicio 7 · crear_evento guarda el evento", lambda: crear_evento_directo() == 1, "Completa el cuerpo de crear_evento en agente.py")
revisar("Ejercicio 7 · el agente tiene las cuatro tools", lambda: len(tools_del_archivo()) == 4, "Registra consultar_agenda, buscar_correos, clima y crear_evento en crear_agente")
revisar("Ejercicio 7 · escribir exige aprobación", lambda: tools_del_archivo()["crear_evento"].requires_approval, "Envuelve crear_evento con Tool(..., requires_approval=True)")


## Evidencia y cierre
Guarda en tu entrega: tu `minutos_libres_dia`, la tool `buscar_correos_bien`, tu `normalizar_hora`, tu `probar_aprobacion` y tu carpeta `mi_copiloto/` completada.

**¿Qué puede hacer ahora Mi copiloto que antes no podía?** Consulta agenda, correo y clima, **cita sus fuentes**, se corrige con `ModelRetry` y **pide permiso antes de escribir**. Una pregunta para llevarte: ¿qué decidiste **no** dejarle hacer?

**Siguiente (opcional, con IA real):** `03_ia_real.ipynb`: Ejercicio 8 (usar, preguntar o negarse) y Ejercicio 9 (reto: el mismo pedido con dos modelos).
